# 01 Data Exploration
Our dataset is LendingClub data for accepted loans between 2007 and 2018. LendingClub is an American digital marketplace bank listed on the New York Stock Exchange (LC), founded in 2006 and specialising in peer-to-peer lending. This dataset is available at https://www.kaggle.com/datasets/wordsforthewise/lending-club?select=accepted_2007_to_2018Q4.csv.gz under a CC0: Public Domain licence.

We begin with a standard initialising of the dataframe and a summarising of its contents.

In [1]:
import pandas as pd

#load
df = pd.read_csv(
    #dataset is LendingClub accepted loan data from 2007 to 2018
    #available on Kaggle https://www.kaggle.com/datasets/wordsforthewise/lending-club?select=accepted_2007_to_2018Q4.csv.gz
    'data/accepted_2007_to_2018Q4.csv',
    low_memory=False
)
#sample
df = df.sample(n=100000, random_state=42)

#print some common attributes
print(df.shape)       #number of rows and columns
print(df.head())      #first few rows
print(df.info())      #column names, data types, null counts
print(df.describe())  #summary statistics

(100000, 151)
                id  member_id  loan_amnt  funded_amnt  funded_amnt_inv  \
392949    39651438        NaN    32000.0      32000.0          32000.0   
1273506   16411620        NaN     9600.0       9600.0           9600.0   
324024    45122316        NaN     4000.0       4000.0           4000.0   
2066630  125356772        NaN     6025.0       6025.0           6025.0   
477199   128490686        NaN    25000.0      25000.0          25000.0   

               term  int_rate  installment grade sub_grade  ...  \
392949    60 months     10.49       687.65     B        B3  ...   
1273506   36 months     12.99       323.42     C        C1  ...   
324024    36 months      6.68       122.93     A        A3  ...   
2066630   36 months     10.91       197.00     B        B4  ...   
477199    60 months     26.30       752.96     E        E5  ...   

        hardship_payoff_balance_amount hardship_last_payment_amount  \
392949                             NaN                          NaN

### Missing Data
Before we begin to explore our dataframe, we must understand the scale of missing data. The below code gives us a table showing how many rows are missing data in each column.

In [2]:
#compute amount of missing data per column and calculate percentage of missing data per column
missing = df.isnull().sum()
missing_pct = (missing / len(df)).round(3)


missing_df = pd.DataFrame({'missing_count': missing, 'missing_percentage': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_percentage', ascending=False))

                                            missing_count  missing_percentage
member_id                                          100000               1.000
orig_projected_additional_accrued_interest          99617               0.996
hardship_payoff_balance_amount                      99510               0.995
hardship_last_payment_amount                        99510               0.995
payment_plan_start_date                             99510               0.995
...                                                   ...                 ...
last_pymnt_amnt                                         4               0.000
tax_liens                                               5               0.000
hardship_flag                                           4               0.000
disbursement_method                                     4               0.000
debt_settlement_flag                                    4               0.000

[150 rows x 2 columns]


## Defaulted Loans
Of interest to us are loans that have been defaulted. We shall use this historical data to build a forward-looking model to estimate the expected loss from loans across a given year.

First, we list all columns of the dataframe. We are looking for those that indicate the status of the loan, i.e. whether the loan was paid, in-progress or the borrower defaulted.

In [3]:
print(df.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

Of note in the above output are `loan_status` and `settlement_status`. We review a random sample from these columns.

In [4]:
df[['id', 'loan_status', 'settlement_status']].sample(5)

,id,loan_status,settlement_status
662877,80790494,Fully Paid,NaN
1097370,69652617,Fully Paid,NaN
907621,109516724,Current,NaN
2077665,125084655,Fully Paid,NaN
634021,113200143,Fully Paid,NaN


The `loan_status` column is of more relevance to us, based on the output reviewed. As such, we output the unique values in that column to identify those that indicate a default. We also consider rows with missing data, which we remove from our dataframe if the number is insignificant.

In [5]:
#identify number of rows with missing data in the loan_status column, drop these rows from the dataset
missing_loan_status = df['loan_status'].isnull().sum()
df = df.dropna(subset=['loan_status'])
print(f'Number of rows removed due to missing loan status data: {missing_loan_status}')

#output unique values in the loan status column
unique_values_loan_status = df['loan_status'].unique()
print(f'Unique loan status values: {unique_values_loan_status}')

Number of rows removed due to missing loan status data: 4
Unique loan status values: ['Current' 'Fully Paid' 'Charged Off' 'Late (31-120 days)'
 'Late (16-30 days)' 'In Grace Period'
 'Does not meet the credit policy. Status:Fully Paid'
 'Does not meet the credit policy. Status:Charged Off']


There is expected to be a inconsequential number of rows with missing `loan_status` data - so we may proceed with our slightly smaller sample.

As we are interested in borrowers that have defaulted on their loan, we must determine which of the above constitute a default for impairment purposes. We go through one-by-one:
- ✗ `Current`: we do not include these loans as they are in-progress and payments have been made on time.
- ✗ `Fully Paid`: these are loans that have been paid off and as such are not included.
- ✓ `Charged Off`: these loans have been written-off. Naturally, these are to be considered defaults for our purposes.
- ✓ `Late (31-120 days)`: these loans have a late payment of over one month. We consider these to be defaults due to the high risk of default.
- ✓ `Late (16-30 days)`: although a marginally lower risk of default than the previous, we still consider the risk of default to be of significance.
- ✗ `In Grace Period`: despite a payment being late, we do not consider loans of this type to be a high enough risk of default to include in our model.
- ✗ `Does not meet the credit policy. Status:Fully Paid`: we treat these loans as we treat standard `Fully Paid` loans.
- ✓ `Does not meet the credit policy. Status:Charged Off`: we treat these loans as we treat standard `Charged Off` loans.

We then define our `default_flag` and print the number of flagged rows. We also calculate the default rate to three decimal places.

In [6]:
#loan statuses indicating a default
default_statuses = [
    'Charged Off',
    'Late (31-120 days)',
    'Late (16-30 days)',
    'Does not meet the credit policy. Status:Charged Off'
]
df['default_flag'] = df['loan_status'].isin(default_statuses).astype(int)

#count default flag and print
print(df['default_flag'].value_counts())

#calculate and print default rate
default_rate = df['default_flag'].mean()
print(f'Default Rate: {default_rate:.3f}')

default_flag
0    86775
1    13221
Name: count, dtype: int64
Default Rate: 0.132


Finally, we save our dataframe.

In [7]:
#save to csv file
df.to_csv('data/sample.csv', index=True)